# Who observed which cell: the observation record

This notebook illustrates how the simulator records, for every cell, whether it was observed, how many times, and which robot observed it first. The value of information (VoI) score needs this to credit the reward for a new observation to the robot that made it.

There are two parts:

1. `simulate_1day` tags every observation with the name of the robot that made it (`observation["robot"]`).
2. Every Waterberry estimator (`WaterberryFarmInformationModel`) holds an `ObservationRecord` in `estimator.record`, updated in `add_observation`.

The setup is a Miniberry-10 farm (11 x 11 cells) with three robots whose paths intersect.

In [ ]:
import sys
import tempfile
sys.path.append("..")

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import ListedColormap

from policy import FollowPathPolicy
from robot import Robot
from water_berry_farm import MiniberryFarm, WaterberryFarmEnvironment, WBF_IM_DiskEstimator, WBF_Score_WeightedAsymmetric
from wbf_simulate import simulate_1day

## Setup: a Miniberry-10 farm and three robots with intersecting paths

- **alpha** goes along the row y=5 to the right edge, then comes back to x=2. It therefore re-observes cells that it has already seen.
- **beta** goes up the column x=5. It reaches the crossing (5,5) in the **same timestep** as alpha.
- **gamma** goes diagonally from (0,1) to (10,9). It passes (5,5) two timesteps after alpha and beta.

A robot that has finished its path stays in place and keeps observing the same cell.

In [ ]:
farm = MiniberryFarm(scale=1)
farm.create_type_map()
environment = WaterberryFarmEnvironment(farm, use_saved=False, seed=10, savedir=tempfile.mkdtemp())
environment.proceed(6)  # like time-start-environment in the 1robot1day expruns

paths = {
    "alpha": [[0, 5], [10, 5], [2, 5]],
    "beta": [[5, 0], [5, 10]],
    "gamma": [[0, 1], [10, 9]],
}
robots = []
for name, path in paths.items():
    robot = Robot(name, path[0][0], path[0][1], 0)
    robot.assign_policy(FollowPathPolicy(1, path))
    robots.append(robot)

estimator = WBF_IM_DiskEstimator(farm.width, farm.height)
results = simulate_1day(environment=environment, robots=robots, estimator=estimator,
                        evaluator=WBF_Score_WeightedAsymmetric(), timesteps=20, estimator_interval=5)

## 1. Every observation is tagged with its robot

`results["observations"]` is indexed `[timestep][robot]`, with the robots in name order. Each observation now carries a `"robot"` field. At timestep 5, alpha and beta are both at (5,5).

In [ ]:
for timestep, observations in enumerate(results["observations"][:9]):
    print(timestep, [(obs["robot"], obs["x"], obs["y"]) for obs in observations])

## 2. The `ObservationRecord` API

`estimator.record` stores only the cells that were observed, as a sparse dict keyed by `(x, y)`. The full Waterberry grid has about 20 M cells, but a run observes only a few thousand of them.

Point queries:
- `count(x, y)`: the number of observations of the cell, 0 if it was never observed.
- `first(x, y)`: the `(robot, time)` of the first observation, or `None` if the cell was never observed.

In [ ]:
record = estimator.record
for cell in [(5, 5), (6, 5), (0, 0)]:
    print(cell, "count:", record.count(*cell), "first:", record.first(*cell))

The crossing (5,5) was observed 4 times: by alpha and beta at t=5, by gamma at t=7, and by alpha again at t=15 on its way back. Alpha is credited with the first observation because of the tie rule: when several robots observe a new cell in the same timestep, the one first in name order wins, since `simulate_1day` adds observations in that order.

Cell (6,5) is a plain crossing without a tie: alpha passed it at t=6, before gamma at t=8.

**Summary query:** `first_counts()` gives the number of cells each robot observed first, i.e. its share of the new information.

In [ ]:
print("observed cells:", len(record.cells), "of", farm.width * farm.height)
print("first observer counts:", record.first_counts())

**Set queries:** `indices(robot=None, since=None, min_count=1)` returns index arrays `(xs, ys)` that index the numpy maps directly, as in `env.tylcv.value[xs, ys]`. The filters can be combined:
- `robot`: only cells first observed by this robot.
- `since`: only cells first observed at or after this time. This gives the new cells of a scoring interval, i.e. the incremental VoI.
- `min_count`: only cells observed at least this many times. With 2, this gives the re-observations.

`mask(**filters)` returns the same selection as a dense boolean `(width, height)` array.

In [ ]:
def as_cells(indices):
    return sorted((int(x), int(y)) for x, y in zip(*indices))

print("first observed by gamma:", as_cells(record.indices(robot="gamma")))
print("new cells in t >= 10:", as_cells(record.indices(since=10)))
print("observed more than once:", as_cells(record.indices(min_count=2)))
print("beta's cells as a mask, sum:", record.mask(robot="beta").sum())

**Using the index arrays on a map:** the same arrays index any `(width, height)` map. For example, this counts the TYLCV- and CCR-diseased cells (ground truth value below 1.0) that each robot observed first. This is the quantity absolute VoI credits with `v_pos`:

In [ ]:
for name in paths:
    xs, ys = record.indices(robot=name)
    print(name, "first-observed cells:", len(xs),
          "TYLCV diseased:", int(np.sum(environment.tylcv.value[xs, ys] < 1.0)),
          "CCR diseased:", int(np.sum(environment.ccr.value[xs, ys] < 1.0)))

## 3. Visualization

- **Left:** the number of observations of each cell. The crossing, alpha's way back, and the cells where robots parked after finishing their paths all have counts above 1.
- **Right:** which robot observed each cell first. The paths are drawn on top.

In [ ]:
names = sorted(paths)
colors = ["tab:blue", "tab:orange", "tab:green"]
counts = np.zeros((farm.width, farm.height), dtype=int)
first = np.full((farm.width, farm.height), np.nan)
for (x, y), rec in record.cells.items():
    counts[x, y] = rec["count"]
    first[x, y] = names.index(rec["first-robot"])

fig, (ax_count, ax_first) = plt.subplots(1, 2, figsize=(11, 5))
image = ax_count.imshow(counts.T, origin="lower", cmap="Greys")
for (x, y), rec in record.cells.items():
    ax_count.text(x, y, rec["count"], ha="center", va="center", color="red", fontsize=8)
fig.colorbar(image, ax=ax_count, shrink=0.8)
ax_count.set_title("record.count(x, y)")

ax_first.imshow(first.T, origin="lower", cmap=ListedColormap(colors), vmin=-0.5, vmax=len(names) - 0.5)
for name, color in zip(names, colors):
    positions = [obs for observations in results["observations"] for obs in observations if obs["robot"] == name]
    ax_first.plot([o["x"] for o in positions], [o["y"] for o in positions], color="black", linewidth=0.8)
    ax_first.plot([], [], "s", color=color, label=f"{name} ({record.first_counts()[name]} cells)")
ax_first.legend(loc="upper left", fontsize=8)
ax_first.set_title("record.first(x, y): first observer")
plt.show()